# PhysicsConv3D Global-Voigt: In-vivo-Spektren und Parameter-Maps

Dieses Notebook evaluiert den separaten globalen Voigt-Run mit 15 Metabolitenamplituden, globalem Frequenzshift, globalem Lorentz-/Gauss-FWHM, zwei Phasenparametern und der stark krümmungsregularisierten komplexen 9-Spline-forD-Baseline. Es lädt immer `last.pt`.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_project_root(start: Path, name: str) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if candidate.name == name and (candidate / 'src').is_dir():
            return candidate
        sibling = candidate / name
        if (sibling / 'src').is_dir():
            return sibling
    raise FileNotFoundError(name)


ROOT = find_project_root(Path.cwd(), 'Denoising')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from denoising.config.build import build_config
from denoising.config.load import load_yaml

# A notebook kernel may still hold the model implementation from an older
# training/evaluation run. Reload in dependency order so inference always
# uses the current scaled parameterization from the source tree.
import denoising.models.physics.parameterization as parameterization_module
import denoising.models.physics.physics_conv3d as physics_conv3d_module
import denoising.models.factory as model_factory_module
importlib.reload(parameterization_module)
importlib.reload(physics_conv3d_module)
importlib.reload(model_factory_module)
build_model = model_factory_module.build_model

print('Denoising:', ROOT)

## Einstellungen

Das Notebook verwendet immer `last.pt`, damit während des Trainings der neueste vollständig gespeicherte Epochenstand betrachtet wird.

In [ ]:
RUN_NAME = 'MS_180_Phive_GlobalVoigt_Sigma_AllNuisanceZReg001_OldStats_NoGlcNoTwoHG'
CHECKPOINT_NAME = 'last.pt'  # immer der neueste abgeschlossene Epochenstand
SUBJECT = 'MS_180'
Z_SLICE = 14
GPU_NUMBER = 0

# Der komplette 64x64-Slice wird als ein Patch ausgewertet.
PATCH_SIZE = 64
PATCH_STRIDE = 64
INFERENCE_BATCH_SIZE = 1
VOXEL_XY = (32, 32)

DEVICE = torch.device(f'cuda:{GPU_NUMBER}' if torch.cuda.is_available() else 'cpu')
RUN_DIR = ROOT / 'trained_models' / RUN_NAME
CONFIG_PATH = RUN_DIR / 'train_physics_7T_phive.yaml'
CHECKPOINT_PATH = RUN_DIR / 'checkpoints' / CHECKPOINT_NAME
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'Run config not found: {CONFIG_PATH}')
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f'No global-Voigt checkpoint yet: {CHECKPOINT_PATH}. '
        'Wait until the first training epoch has completed.'
    )
print('Device:    ', DEVICE)
print('Config:    ', CONFIG_PATH)
print('Checkpoint:', CHECKPOINT_PATH)


## Config, Subject und Modell laden

In [ ]:
cfg = build_config(load_yaml(CONFIG_PATH))
if cfg.run.name != RUN_NAME:
    raise RuntimeError(f'Config run {cfg.run.name!r} does not match {RUN_NAME!r}.')
if cfg.data.spatial_mask_filename is None:
    raise RuntimeError('This checkpoint config has no spatial brain mask.')
data_path = ROOT / cfg.data.base_dir / SUBJECT / cfg.data.data_filename
mask_path = ROOT / cfg.data.base_dir / SUBJECT / cfg.data.spatial_mask_filename

fid_volume = np.load(data_path).astype(np.complex64, copy=False)
brain_mask = np.load(mask_path).astype(bool)
if fid_volume.ndim != 4 or fid_volume.shape[-1] != 558:
    raise ValueError(f'Unexpected data shape: {fid_volume.shape}')
if not 0 <= Z_SLICE < fid_volume.shape[2]:
    raise IndexError(f'Z_SLICE must be in [0, {fid_volume.shape[2] - 1}]')

# Identical to load_and_preprocess_data: one scale per complete subject,
# calculated in FID domain before FFT.
normalization_scale = float(np.max(np.abs(fid_volume))) if cfg.data.normalization else 1.0
normalized_fids = fid_volume / normalization_scale if normalization_scale > 0 else fid_volume
spectra_volume = np.fft.fftshift(
    np.fft.fft(normalized_fids, axis=3), axes=3
).astype(np.complex64)
input_slice = spectra_volume[:, :, Z_SLICE, :]
mask_slice = brain_mask[:, :, Z_SLICE]

sample_shape = (2, PATCH_SIZE, PATCH_SIZE, input_slice.shape[-1])
model = build_model(cfg, sample_shape).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state'], strict=True)
model.eval()

print('Data:       ', data_path)
print('FID shape:  ', fid_volume.shape)
print('Slice shape:', input_slice.shape)
print('FID scale:  ', normalization_scale)
print('Epoch:      ', checkpoint.get('epoch'))
print('Val loss:   ', checkpoint.get('val_loss'))
print('Basis:      ', model.physical_decoder.basis_fids.shape)
print('Lineshape:  ', model.lineshape_model)
print('Parameters: ', model.parameterization.n_output_parameters)
n_basis = model.physical_decoder.n_basis_components
print('Lorentz Z:  ', float(model.parameterization.parameter_mean[n_basis + 1]), '+/-', float(model.parameterization.parameter_std[n_basis + 1]), 'Hz')
print('Gaussian Z: ', float(model.parameterization.parameter_mean[n_basis + 2]), '+/-', float(model.parameterization.parameter_std[n_basis + 2]), 'Hz')
print('Scaling:    ', model.parameterization.__class__.__name__)

## Patchweise Inference

Überlappende Patches werden gleich gewichtet gemittelt. Neben dem rekonstruierten Spektrum werden die bereits physikalisch transformierten Parameter gesammelt.

In [ ]:
def patch_starts(size, patch_size, stride):
    starts = list(range(0, max(size - patch_size + 1, 1), stride))
    last = max(size - patch_size, 0)
    if not starts or starts[-1] != last:
        starts.append(last)
    return starts


size_x, size_y, n_frequency = input_slice.shape
x_starts = patch_starts(size_x, PATCH_SIZE, PATCH_STRIDE)
y_starts = patch_starts(size_y, PATCH_SIZE, PATCH_STRIDE)
locations = [(x, y) for x in x_starts for y in y_starts]
n_basis = model.physical_decoder.n_basis_components

reconstruction_sum = np.zeros_like(input_slice, dtype=np.complex64)
baseline_sum = np.zeros_like(input_slice, dtype=np.complex64)
amplitude_sum = np.zeros((size_x, size_y, n_basis), dtype=np.float32)
global_parameter_sum = {
    'Frequency shift [Hz]': np.zeros((size_x, size_y), np.float32),
    'Lorentz FWHM [Hz]': np.zeros((size_x, size_y), np.float32),
    'Gaussian FWHM [Hz]': np.zeros((size_x, size_y), np.float32),
    'Phase 0 [rad]': np.zeros((size_x, size_y), np.float32),
    'Phase 1 [rad/Hz]': np.zeros((size_x, size_y), np.float32),
}
weight_sum = np.zeros((size_x, size_y), dtype=np.float32)

with torch.inference_mode():
    for start in range(0, len(locations), INFERENCE_BATCH_SIZE):
        batch_locations = locations[start:start + INFERENCE_BATCH_SIZE]
        patches = [
            input_slice[x:x + PATCH_SIZE, y:y + PATCH_SIZE]
            for x, y in batch_locations
        ]
        complex_batch = np.stack(patches)
        network_batch = torch.from_numpy(
            np.stack((complex_batch.real, complex_batch.imag), axis=1)
        ).to(DEVICE)
        output = model(network_batch, return_parameters=True)
        reconstruction = (
            output.reconstruction[:, 0] + 1j * output.reconstruction[:, 1]
        ).cpu().numpy()
        parameters = output.parameters
        baseline_real = (
            parameters.baseline_coefficients_real
            @ model.ford_baseline_design_matrix.T
        )
        baseline_imag = (
            parameters.baseline_coefficients_imag
            @ model.ford_baseline_design_matrix.T
        )
        if model.baseline_conjugate_subject_signals:
            baseline_imag = -baseline_imag
        baseline_batch = (baseline_real + 1j * baseline_imag).cpu().numpy()
        amplitudes = parameters.amplitudes.cpu().numpy()
        global_parameter_batches = [
            parameters.frequency_shift_hz.cpu().numpy(),
            parameters.lorentzian_fwhm_hz.cpu().numpy(),
            parameters.gaussian_fwhm_hz.cpu().numpy(),
            parameters.zero_order_phase_radians.cpu().numpy(),
            parameters.first_order_phase_rad_per_hz.cpu().numpy(),
        ]
        for index, (x, y) in enumerate(batch_locations):
            area = np.s_[x:x + PATCH_SIZE, y:y + PATCH_SIZE]
            reconstruction_sum[area] += reconstruction[index]
            baseline_sum[area] += baseline_batch[index]
            amplitude_sum[area] += amplitudes[index]
            for name, values in zip(global_parameter_sum, global_parameter_batches):
                global_parameter_sum[name][area] += values[index]
            weight_sum[area] += 1.0

reconstruction_slice = reconstruction_sum / weight_sum[..., None]
baseline_slice = baseline_sum / weight_sum[..., None]
amplitude_maps = amplitude_sum / weight_sum[..., None]
global_parameter_maps = {
    name: values / weight_sum for name, values in global_parameter_sum.items()
}
residual_slice = input_slice - reconstruction_slice
masked_input = input_slice[mask_slice]
masked_reconstruction = reconstruction_slice[mask_slice]
masked_residual = residual_slice[mask_slice]
input_rms = float(np.sqrt(np.mean(np.abs(masked_input) ** 2)))
reconstruction_rms = float(np.sqrt(np.mean(np.abs(masked_reconstruction) ** 2)))
masked_complex_mse = float(np.mean(np.abs(masked_residual) ** 2))
print(f'Patches: {len(locations)}, inference complete')
print('Reconstruction:', reconstruction_slice.shape)
print('Amplitude maps:', amplitude_maps.shape)
print(f'Brain-mask input RMS:          {input_rms:.6g}')
print(f'Brain-mask reconstruction RMS: {reconstruction_rms:.6g}')
output_to_input_rms = reconstruction_rms / input_rms if input_rms > 0 else np.nan
zero_solution_mse = float(np.mean(np.abs(masked_input) ** 2))
print(f'Brain-mask complex MSE:         {masked_complex_mse:.6g}')
print(f'Zero-solution complex MSE:      {zero_solution_mse:.6g}')
print(f'Output / input RMS:             {output_to_input_rms:.4f}')
if output_to_input_rms < 0.1:
    print('WARNING: Reconstruction energy is below 10% of the input; possible zero collapse.')


## Input, Rekonstruktion und Residuum an einem Voxel

In [ ]:
voxel_x, voxel_y = VOXEL_XY
if not mask_slice[voxel_x, voxel_y]:
    coordinates = np.argwhere(mask_slice)
    center = np.array([size_x / 2, size_y / 2])
    voxel_x, voxel_y = coordinates[np.argmin(np.sum((coordinates - center) ** 2, axis=1))]

input_spectrum = input_slice[voxel_x, voxel_y]
fitted_spectrum = reconstruction_slice[voxel_x, voxel_y]
baseline_spectrum = baseline_slice[voxel_x, voxel_y]
residual_spectrum = residual_slice[voxel_x, voxel_y]
frequency_hz = np.fft.fftshift(np.fft.fftfreq(
    n_frequency, d=model.physical_decoder.dwell_time_seconds
))

fit_mask = model.denoising_frequency_mask.detach().cpu().numpy().astype(bool)
outside_fit_mask = ~fit_mask

fig, axes = plt.subplots(1, 3, figsize=(22, 5.5), constrained_layout=True)
axes[0].plot(frequency_hz, input_spectrum.real, label='Input', lw=1.2)
axes[0].plot(frequency_hz, fitted_spectrum.real, label='Physics reconstruction', lw=1.2)
axes[0].plot(frequency_hz, np.where(fit_mask, baseline_spectrum.real, np.nan), '--', label='forD baseline', lw=1.5, alpha=0.9)
axes[1].plot(frequency_hz, input_spectrum.imag, label='Input', lw=1.2)
axes[1].plot(frequency_hz, fitted_spectrum.imag, label='Physics reconstruction', lw=1.2)
axes[1].plot(frequency_hz, np.where(fit_mask, baseline_spectrum.imag, np.nan), '--', label='forD baseline', lw=1.5, alpha=0.9)
axes[2].plot(frequency_hz, np.where(fit_mask, residual_spectrum.real, np.nan), label='Residual real (fit range)')
axes[2].plot(frequency_hz, np.where(fit_mask, residual_spectrum.imag, np.nan), label='Residual imaginary (fit range)', alpha=0.8)
axes[2].plot(frequency_hz, np.where(outside_fit_mask, input_spectrum.real, np.nan), label='Input real (outside fit range)', lw=1.0, alpha=0.75)
axes[2].plot(frequency_hz, np.where(outside_fit_mask, input_spectrum.imag, np.nan), label='Input imaginary (outside fit range)', lw=1.0, alpha=0.75)
titles = ['Real', 'Imaginary', 'Input − reconstruction / input outside fit range']
for ax, title in zip(axes, titles):
    ax.set_title(title)
    ax.set_xlabel('Frequency offset [Hz]')
    ax.grid(alpha=0.2)
    ax.legend()
fig.suptitle(f'{SUBJECT}, z={Z_SLICE}, voxel=({voxel_x}, {voxel_y}), epoch={checkpoint.get("epoch")}', fontsize=16)
plt.show()

## Amplituden-Maps aller Basisbestandteile

Jede Map verwendet ihre eigene robuste Skala innerhalb der Hirnmaske (1.–99. Perzentil). Angezeigt werden die positiven physikalischen Amplituden (`softplus(raw)`), nicht die internen Raw-Outputs. Die Werte beziehen sich auf die subjectweise normalisierten Trainingsdaten.

In [ ]:
basis_names = tuple(model.basis_names)

sum_groups = {
    'tNAA': ('NAA', 'NAAG'),
    'tCr': ('Cr', 'PCr'),
    'Glx': ('Glu', 'Gln'),
}

display_names = {
    'tNAA': 'tNAA (NAA + NAAG)',
    'tCr': 'tCr (Cr + PCr)',
    'Glx': 'Glx (Glu + Gln)',
}

maps = {
    name: amplitude_maps[..., index]
    for index, name in enumerate(basis_names)
}

for total, components in sum_groups.items():
    maps[total] = sum(maps[name] for name in components)

plot_names = list(basis_names) + list(sum_groups)

n_columns = 5
n_rows = int(np.ceil(len(plot_names) / n_columns))

fig, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(20, 3.6 * n_rows),
    constrained_layout=True,
    squeeze=False,
)

for name, ax in zip(plot_names, axes.flat):
    values = maps[name]
    inside = values[mask_slice & np.isfinite(values)]

    vmin, vmax = (
        np.percentile(inside, [1, 99])
        if inside.size
        else (0, 1)
    )

    if vmax <= vmin:
        vmax = vmin + 1e-6

    display = np.where(mask_slice, values, np.nan)

    image = ax.imshow(
        display.T,
        origin='lower',
        cmap='magma',
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(display_names.get(name, name))
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.03)

for ax in axes.flat[len(plot_names):]:
    ax.axis('off')

fig.suptitle(
    f'Metabolite amplitude maps with forD baseline — '
    f'{SUBJECT}, z={Z_SLICE}',
    fontsize=18,
)

plt.show()

## Globale Voigt- und Phasenparameter

Gezeigt werden die fünf globalen Parameter pro Voxel. Lorentz- und Gauss-FWHM werden vom Netz in z-standardisierten Koordinaten vorhergesagt und im festen Decoder nach Hz zurücktransformiert. Es existieren weder ein 23-Bin-Kernel noch metabolitenspezifische Shift-/FWHM-Maps.


In [ ]:
fig, axes = plt.subplots(1, len(global_parameter_maps), figsize=(22, 4.5), constrained_layout=True)
for ax, (name, values) in zip(np.atleast_1d(axes), global_parameter_maps.items()):
    inside = values[mask_slice & np.isfinite(values)]
    vmin, vmax = (np.percentile(inside, [1, 99]) if inside.size else (0, 1))
    if vmax <= vmin:
        vmax = vmin + 1e-6
    image = ax.imshow(
        np.where(mask_slice, values, np.nan).T,
        origin='lower', cmap='coolwarm', vmin=vmin, vmax=vmax,
    )
    ax.set_title(name)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.03)
fig.suptitle(f'Global Voigt and phase parameters — {SUBJECT}, z={Z_SLICE}', fontsize=17)
plt.show()

print('Formal parameter count per voxel:', model.parameterization.n_output_parameters)
print('  Metabolite amplitudes:', n_basis)
print('  Global shift/Voigt/phase:', 5)
print('  Complex baseline coefficients:', 2 * model.parameterization.baseline_n_splines)


## Direkter Vergleich: klassischer forD-Fit und Physics-Denoising

In [ ]:
import json

# Nur diesen Pfad anpassen, falls der laufende forD-Fit anders heißt.
FORD_OUTPUT_DIR = (
    ROOT.parent / 'walinet' / 'data' / '7T' / 'B0corrected_wo_LipidMask'
    / SUBJECT / 'MetabMapsAfterWalinet_Final'
)
FORD_MAP_PATH = FORD_OUTPUT_DIR / 'metabolite_maps' / 'metabolite_maps.npy'
FORD_METADATA_PATH = FORD_OUTPUT_DIR / 'parameter_maps_metadata.json'
MAGNITUDE_PATH = data_path.parent / 'magnitude.npy'

if not FORD_MAP_PATH.is_file():
    raise FileNotFoundError(
        f'forD map is not available yet: {FORD_MAP_PATH}\n'
        'The fitting job may still be running. Re-run this cell after export.'
    )
if not FORD_METADATA_PATH.is_file():
    raise FileNotFoundError(f'forD metadata is missing: {FORD_METADATA_PATH}')
if not MAGNITUDE_PATH.is_file():
    raise FileNotFoundError(f'Magnitude image is missing: {MAGNITUDE_PATH}')

ford_maps = np.load(FORD_MAP_PATH, allow_pickle=False)
magnitude_volume = np.load(MAGNITUDE_PATH, allow_pickle=False)
with FORD_METADATA_PATH.open('r', encoding='utf-8') as handle:
    ford_metadata = json.load(handle)
ford_names = tuple(ford_metadata['metabolite_axis_order'])

# A legacy export can retain a singleton repetition axis.
if ford_maps.ndim == 5 and ford_maps.shape[0] == 1:
    ford_maps = ford_maps[0]
expected_spatial_shape = tuple(fid_volume.shape[:3])
if ford_maps.shape[:3] != expected_spatial_shape:
    raise ValueError(
        f'forD spatial shape {ford_maps.shape[:3]} does not match '
        f'input shape {expected_spatial_shape}.'
    )
if ford_maps.shape[-1] != len(ford_names):
    raise ValueError(
        f'forD map has {ford_maps.shape[-1]} channels, but metadata lists '
        f'{len(ford_names)} metabolites.'
    )
if magnitude_volume.shape[:3] != expected_spatial_shape:
    raise ValueError(
        f'Magnitude shape {magnitude_volume.shape[:3]} does not match '
        f'input shape {expected_spatial_shape}.'
    )

ford_index = {name: index for index, name in enumerate(ford_names)}
physics_index = {name: index for index, name in enumerate(basis_names)}
common_names = tuple(name for name in basis_names if name in ford_index)
physics_only = tuple(name for name in basis_names if name not in ford_index)
ford_only = tuple(name for name in ford_names if name not in physics_index)
if not common_names:
    raise ValueError('forD and Physics outputs have no common metabolites.')
if physics_only:
    print('Skipped Physics-only metabolites:', ', '.join(physics_only))
if ford_only:
    print('Skipped forD-only metabolites:', ', '.join(ford_only))

n_rows = len(common_names)
fig, axes = plt.subplots(
    n_rows, 3, figsize=(14, 3.7 * n_rows), constrained_layout=True, squeeze=False
)
magnitude_slice = magnitude_volume[:, :, Z_SLICE]
for row_index, name in enumerate(common_names):
    ford_slice = ford_maps[:, :, Z_SLICE, ford_index[name]]
    denoised_slice = amplitude_maps[..., physics_index[name]]

    for column, values, method, cmap, restrict_to_brain in (
        (0, magnitude_slice, 'Magnitude', 'gray', False),
        (1, ford_slice, 'no Denoising', 'magma', True),
        (2, denoised_slice, 'Denoised', 'magma', True),
    ):
        # forD and Physics use differently scaled basis sets. Therefore each
        # map gets its own robust color scale; compare spatial contrast, not
        # the absolute color values between columns.
        scale_mask = np.isfinite(values)
        if restrict_to_brain:
            scale_mask &= mask_slice
        inside = values[scale_mask]
        vmin, vmax = (
            np.percentile(inside, [1, 99]) if inside.size else (0.0, 1.0)
        )
        if vmax <= vmin:
            vmax = vmin + 1e-6
        display = np.where(mask_slice, values, np.nan) if restrict_to_brain else values
        image = axes[row_index, column].imshow(
            display.T, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax
        )
        axes[row_index, column].set_title(f'{name} — {method}')
        axes[row_index, column].set_xticks([])
        axes[row_index, column].set_yticks([])
        fig.colorbar(image, ax=axes[row_index, column], fraction=0.046, pad=0.03)

fig.suptitle(
    f'Magnitude (left), forD fit (center), Physics denoised (right) — {SUBJECT}, z={Z_SLICE}',
    fontsize=18,
)
SAVE_PATH = FORD_OUTPUT_DIR / f'ford_vs_physics_{SUBJECT}_z{Z_SLICE:02d}.png'

fig.savefig(
    SAVE_PATH,
    dpi=200,
    bbox_inches='tight',
    facecolor='white',
)
print('Saved:', SAVE_PATH)

plt.show()
print('forD maps:', FORD_MAP_PATH)
print('Compared common metabolites:', ', '.join(common_names))